---
title: "深度优先搜索 (DFS)——剪枝&记忆化"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [11]:
#| code-fold: true
from typing import List, Optional
import copy

## 📝 题目 51：N 皇后

::: {.callout-caution}
### 📖 题目描述
按照国际象棋的规则，皇后可以攻击与她在同一行、同一列或同一斜线上的棋子。

**n 皇后问题** 研究的是如何将 `n` 个皇后放置在 `n × n` 的棋盘上，并且使皇后彼此之间不能相互攻击。

给你一个整数 `n` ，返回所有不同的 **n 皇后问题** 的解决方案。

**每一种解法包含一个不同的棋局**，其中 `'Q'` 和 `'.'` 分别代表了皇后和空位。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 4`
    * **输出**：`[[".Q..","...Q","Q...","..Q."],["..Q.","Q...","...Q",".Q.."]]`
    * **解释**：4 皇后问题存在两个不同的解。

* **示例 2**：
    * **输入**：`n = 1`
    * **输出**：`[["Q"]]`
:::

In [5]:
class Solution51:
    def solveNQueens(self, n: int) -> List[List[str]]:
        result = []
        board = [['.' for _ in range(n)] for _ in range(n)]  # 初始化棋盘，全部填入 '.'
        used_cols = set()
        used_dig1 = set()  # 主对角线 (row - col)
        used_dig2 = set()  # 副对角线 (row + col)

        def backtrack(row: int) -> None:
            if row == n:  # 终点：如果填满了最后一行
                result.append(''.join(r) for r in board)
                return
            for col in range(n):
                if col in used_cols or (row - col) in used_dig1 or (row + col) in used_dig2:  # 检查是否冲突
                    continue
                board[row][col] = 'Q'  # 做选择
                used_cols.add(col)
                used_dig1.add(row - col)
                used_dig2.add(row + col)
                backtrack(row + 1)  # 进入下一行
                board[row][col] = '.'  # 撤销选择
                used_cols.remove(col)
                used_dig1.remove(row - col)
                used_dig2.remove(row + col)

        backtrack(0)
        return result

In [6]:
#| code-fold: true
def test_51(func):
    test_cases = [
        {"n": 1, "exp_count": 1, "desc": "1x1 棋盘"},
        {"n": 2, "exp_count": 0, "desc": "2x2 棋盘 (无解)"},
        {"n": 3, "exp_count": 0, "desc": "3x3 棋盘 (无解)"},
        {"n": 4, "exp_count": 2, "desc": "4x4 棋盘 (经典)"},
        {"n": 5, "exp_count": 10, "desc": "5x5 棋盘"},
        {"n": 6, "exp_count": 4, "desc": "6x6 棋盘"},
        {"n": 7, "exp_count": 40, "desc": "7x7 棋盘"},
        {"n": 8, "exp_count": 92, "desc": "8 皇后 (著名问题)"},
        {"n": 9, "exp_count": 352, "desc": "9x9 棋盘"},
        {"n": 0, "exp_count": 0, "desc": "边界情况测试"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期解数':<8} | {'实际解数':<8} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        if tc["n"] == 0:
            actual = []
        else:
            actual = func(tc["n"])
        is_pass = len(actual) == tc["exp_count"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['exp_count']:<8} | {len(actual):<8} | {tc['desc']}")
    print("-" * 65)
    print(f"总结: 通过 {passed}/10")

# 运行测试
test_51(Solution51().solveNQueens)

ID   | 结果     | 预期解数     | 实际解数     | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 1        | 1        | 1x1 棋盘
2    | ✅ PASS | 0        | 0        | 2x2 棋盘 (无解)
3    | ✅ PASS | 0        | 0        | 3x3 棋盘 (无解)
4    | ✅ PASS | 2        | 2        | 4x4 棋盘 (经典)
5    | ✅ PASS | 10       | 10       | 5x5 棋盘
6    | ✅ PASS | 4        | 4        | 6x6 棋盘
7    | ✅ PASS | 40       | 40       | 7x7 棋盘
8    | ✅ PASS | 92       | 92       | 8 皇后 (著名问题)
9    | ✅ PASS | 352      | 352      | 9x9 棋盘
10   | ✅ PASS | 0        | 0        | 边界情况测试
-----------------------------------------------------------------
总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

N 皇后问题的本质是：**在每一行中选取一个位置放置皇后，且该位置不能与之前放置的皇后冲突。**

1. **决策树结构**：
   - **每一层递归**：代表棋盘的一行（Row）。
   - **每一层中的 for 循环**：代表该行中的每一列（Column）。
   - **路径 (path)**：已经摆放好的棋盘状态。

2. **三个维度的约束**：
   为了保证皇后互不攻击，我们需要检查三个地方：
   - **列冲突**：同一个列（Column）不能有两个皇后。
   - **主对角线冲突**：左上到右下方向。规律是：`row - col` 的值固定。
   - **副对角线冲突**：右上到左下方向。规律是：`row + col` 的值固定。

3. **回溯三部曲**：
   - **做选择**：在 `(row, col)` 放置皇后 `'Q'`，并更新三个维度的“已占用”状态。
   - **递归探索**：进入下一行 `row + 1`。
   - **撤销选择**：将 `(row, col)` 恢复为 `'.'`，并清除“已占用”状态。

4. **状态记录优化**：
   虽然我们可以写一个 `isValid` 函数去检查，但最高效的方法是使用三个集合（Set）或布尔数组来分别记录哪些 **列**、**主对角线**、**副对角线** 已经被占领了。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N!)$。第一行有 $N$ 个位置可选，第二行最多 $N-2$ 个，依此类推。实际上搜索空间比 $N!$ 小得多。
* **空间复杂度**：$O(N)$。主要开销是用于记录状态的集合和递归调用栈。
:::

## 📝 题目 139：单词拆分

::: {.callout-caution}
### 📖 题目描述
给你一个字符串 `s` 和一个字符串列表 `wordDict` 作为字典。请你判断是否可以利用字典中出现的单词拼接出 `s` 。

**注意**：

- 不要求字典中所有的单词都在 `s` 中出现。
- 字典中的单词可以 **重复使用** 。
- 你可以假设字典中没有重复的单词。

**输入输出示例**：

* **示例 1**：
    * **输入**：`s = "leetcode"`, `wordDict = ["leet", "code"]`
    * **输出**：`True`
    * **解释**：返回 `True` 因为 `"leetcode"` 可以被拆分成 `"leet code"`。

* **示例 2**：
    * **输入**：`s = "applepenapple"`, `wordDict = ["apple", "pen"]`
    * **输出**：`True`
    * **解释**：返回 `True` 因为 `"applepenapple"` 可以被拆分成 `"apple pen apple"`。注意你可以重复使用字典中的单词。

* **示例 3**：
    * **输入**：`s = "catsandog"`, `wordDict = ["cats", "dog", "sand", "and", "cat"]`
    * **输出**：`False`
:::

In [9]:
class Solution139:
    def wordBreak(self, s: str, wordDict: List[str]) -> bool:
        wordSet = set(wordDict)  # 转成集合，查找更快
        memo = [None] * len(s)  # memo[i] 表示从索引 i 开始的子串是否可以被拆分

        def backtrack(startIndex: int) -> Optional[bool]:
            if startIndex == len(s):  # 终点：如果已经切到了字符串末尾，说明成功了
                return True
            if memo[startIndex] is not None:  # 记忆化：如果之前算过这个位置，直接返回结果
                return memo[startIndex]
            for i in range(startIndex + 1, len(s) + 1):
                word = s[startIndex:i]  # 尝试从 startIndex 开始切出各种长度的前缀，i 是结束位置
                if word in wordSet and backtrack(i):
                    memo[startIndex] = True
                    return True
            memo[startIndex] = False  # 如果所有长度都试过了都不行，记录并返回 False
            return False

        return backtrack(0)

In [10]:
#| code-fold: true
def test_139(func):
    test_cases = [
        {"s": "leetcode", "dict": ["leet", "code"], "exp": True, "desc": "标准两个词"},
        {"s": "applepenapple", "dict": ["apple", "pen"], "exp": True, "desc": "单词重复使用"},
        {"s": "catsandog", "dict": ["cats", "dog", "sand", "and", "cat"], "exp": False, "desc": "无法匹配末尾"},
        {"s": "apple", "dict": ["apple"], "exp": True, "desc": "整个字符串就是一个词"},
        {"s": "ab", "dict": ["a", "b"], "exp": True, "desc": "两个单字母词"},
        {"s": "aaaaaaa", "dict": ["aaaa", "aaa"], "exp": True, "desc": "重叠长度词"},
        {"s": "ccbb", "dict": ["bc", "cb"], "exp": False, "desc": "顺序错误"},
        {"s": "goalspecial", "dict": ["go", "goal", "special"], "exp": True, "desc": "前缀包含测试"},
        {"s": "bb", "dict": ["a", "b", "bbb"], "exp": True, "desc": "包含多余词"},
        {"s": "aaaaaaaaaaaaaaaaabaa", "dict": ["aa", "aaa", "aaaa"], "exp": False, "desc": "长字符串失败测试"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 45)

    for i, tc in enumerate(test_cases):
        actual = func(tc["s"], tc["dict"])
        is_pass = actual == tc["exp"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")

    print("-" * 45)
    print(f"总结: 通过 {passed}/10")

# 运行测试
test_139(Solution139().wordBreak)

ID   | 结果     | 描述
---------------------------------------------
1    | ✅ PASS | 标准两个词
2    | ✅ PASS | 单词重复使用
3    | ✅ PASS | 无法匹配末尾
4    | ✅ PASS | 整个字符串就是一个词
5    | ✅ PASS | 两个单字母词
6    | ✅ PASS | 重叠长度词
7    | ✅ PASS | 顺序错误
8    | ✅ PASS | 前缀包含测试
9    | ✅ PASS | 包含多余词
10   | ✅ PASS | 长字符串失败测试
---------------------------------------------
总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题可以看作是在字符串上寻找一条“通往终点的路径”。

1. **回溯决策树**：
   - **起点**：字符串的索引 0。
   - **决策**：从当前位置开始，尝试所有可能的切割长度。比如 `s = "leetcode"`，你可以尝试切出 `"l"`, `"le"`, `"lee"`, `"leet"` 等。
   - **合法性检查**：如果切下的前缀在 `wordDict` 中，那么我们就递归地去处理剩下的字符串。

2. **核心陷阱：超时 (TLE)**：
   - 简单的回溯会面临巨大的重复计算。比如 `s = "aaaaaaaab"`，字典里有 `["a", "aa", "aaa"]`。你会多次发现后缀 `"aaab"` 无法被拆分。
   - **解决方案：记忆化回溯 (Memoization)**。我们用一个列表或字典 `memo` 记录：从索引 `i` 开始的后缀字符串是否能被成功拆分。如果之前算过，直接返回结果。

3. **回溯三部曲**：
   - **参数**：`startIndex`（当前处理到字符串的哪个位置）。
   - **终止条件**：`startIndex == len(s)`，说明整个字符串都顺利切完了。
   - **横向遍历**：尝试从 `startIndex` 到 `len(s)` 的所有切割点。

4. **优化细节**：
   - 将 `wordDict` 转为 `set`（集合），这样查找单词的时间复杂度从 $O(L)$ 降为 $O(1)$。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n^2)$。其中 $n$ 是字符串长度。有了记忆化，每个起始位置只会计算一次，每次计算涉及一个 $O(n)$ 的循环。
* **空间复杂度**：$O(n)$。记忆化数组的大小和递归栈的深度。
:::

## 📝 题目 79：单词搜索

::: {.callout-caution}
### 📖 题目描述
给定一个 `m x n` 二维字符网格 `board` 和一个字符串单词 `word` 。如果 `word` 存在于网格中，返回 `true` ；否则，返回 `false` 。

单词必须按照字母顺序，通过 **相邻的单元格** 内的字母构成，其中“相邻”单元格是那些水平相邻或垂直相邻的单元格。**同一个单元格内的字母不允许被重复使用。**

**输入输出示例**：

* **示例 1**：
    * **输入**：`board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "ABCCED"`
    * **输出**：`true`

* **示例 2**：
    * **输入**：`board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "SEE"`
    * **输出**：`true`

* **示例 3**：
    * **输入**：`board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "ABCB"`
    * **输出**：`false`
:::

In [16]:
class Solution79:
    def exist(self, board: List[List[str]], word: str) -> bool:
        rows, cols = len(board), len(board[0])

        def backtrack(r: int, c: int, index: int) -> bool:
            if index == len(word):  # 成功：匹配完了单词最后一个字符
                return True
            if not(0 <= r < rows and 0 <= c < cols and word[index] == board[r][c]):  # 失败：越界或字符不匹配
                return False
            temp = board[r][c]  # 做选择
            board[r][c] = '#'  # 标记为已访问
            found = backtrack(r + 1, c, index + 1) or backtrack(r - 1, c, index + 1) or backtrack(r, c + 1, index + 1) or backtrack(r, c - 1, index + 1)  # 只要有一个方向返回 True，结果就是 True
            board[r][c] = temp  # 撤销选择
            return found

        for i in range(rows):
            for j in range(cols):
                if board[i][j] == word[0]:
                    if backtrack(i, j, 0):
                        return True
        return False

In [17]:
#| code-fold: true
def test_79(func):
    test_cases = [
        {"b": [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], "w": "ABCCED", "exp": True},
        {"b": [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], "w": "SEE", "exp": True},
        {"b": [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], "w": "ABCB", "exp": False},
        {"b": [["a","b"],["c","d"]], "w": "abcd", "exp": False, "desc": "不连通的顺序"},
        {"b": [["A","B"],["C","D"]], "w": "ACDB", "exp": True, "desc": "蛇形路径"},
        {"b": [["A"]], "w": "A", "exp": True},
        {"b": [["A"]], "w": "B", "exp": False},
        {"b": [["a","a"]], "w": "aaa", "exp": False, "desc": "长度超标"},
        {"b": [["C","A","A"],["A","A","A"],["B","C","D"]], "w": "AAB", "exp": True},
        {"b": [["A","B","C","E"],["S","F","E","S"],["A","D","E","E"]], "w": "ABCESEEEFS", "exp": True}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 45)
    for i, tc in enumerate(test_cases):
        # 复制 board 防止被修改
        b_copy = copy.deepcopy(tc["b"])
        actual = func(b_copy, tc["w"])
        is_pass = actual == tc["exp"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        desc = tc.get("desc", f"单词: {tc['w']}")
        print(f"{i+1:<4} | {status:<6} | {desc}")

    print("-" * 45)
    print(f"总结: 通过 {passed}/10")

# 运行验证
test_79(Solution79().exist)

ID   | 结果     | 描述
---------------------------------------------
1    | ✅ PASS | 单词: ABCCED
2    | ✅ PASS | 单词: SEE
3    | ✅ PASS | 单词: ABCB
4    | ✅ PASS | 不连通的顺序
5    | ✅ PASS | 蛇形路径
6    | ✅ PASS | 单词: A
7    | ✅ PASS | 单词: B
8    | ✅ PASS | 长度超标
9    | ✅ PASS | 单词: AAB
10   | ✅ PASS | 单词: ABCESEEEFS
---------------------------------------------
总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题是“岛屿问题”的升级版。在岛屿问题中，我们把走过的地方变水就行了；但在单词搜索中，一条路径不通，不代表这个格子不能被另一条路径使用。

1. **寻找入口**：
   - 遍历整个棋盘，只要发现 `board[r][c]` 等于单词的第一个字母，就以此为起点发动“特工潜入”（DFS）。

2. **DFS 潜入逻辑**：
   - **参数**：当前坐标 `(r, c)`，以及当前要匹配单词的第几个字母 `index`。
   - **成功条件**：如果 `index` 达到了单词的长度，说明所有字母都对上了，返回 `True`。
   - **失败条件**：
     - 越界。
     - 当前格子的字母与 `word[index]` 对不上。
     - 当前格子已经在当前路径中被踩过了。

3. **回溯的核心：状态重置**：
   - **做选择**：为了防止“原地转圈”，我们在踏上一个格子时，先把它临时改成一个特殊符号（如 `#`）。
   - **探索**：向上下左右四个方向递归，只要有一个方向能通，就一通百通。
   - **撤销选择 (关键！)**：如果四个方向都走不通，说明这条路是死路。我们要把刚才改成的 `#` **还原回原来的字母**。这样当主循环尝试其他起点，或者其他路径经过这里时，它依然是一个有效的字母。

4. **为什么不用 visited 数组？**
   - 直接修改原数组 `board` 再改回来是一种“原地修改”技巧，可以节省 $O(M \times N)$ 的额外空间。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N \cdot 3^L)$。其中 $L$ 为单词长度。每次搜索有 3 个方向可选（不走回头路）。
* **空间复杂度**：$O(L)$。递归栈的最大深度等于单词的长度。
:::